# TOB予測モデル SHAP分析

論文「機械学習による他社株TOBの予測可能性」の再現 — 特徴量寄与度分析。
`train_rf.py` のキャッシュデータを再利用してモデルを再構築し、SHAP値を計算・可視化する。

In [ ]:
import sys
import subprocess
from pathlib import Path

# --- Runtime detection ---
try:
    from google.colab import auth, userdata
    RUNTIME = 'colab'
except ImportError:
    RUNTIME = 'local'

# --- Colab: install deps & mount Drive ---
if RUNTIME == 'colab':
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q',
         'shap', 'imbalanced-learn', 'japanize-matplotlib', 'structlog', 'optuna'],
        capture_output=True,
    )
    from google.colab import drive
    drive.mount('/content/drive')

# --- Paths ---
if RUNTIME == 'colab':
    PROJECT_ROOT = Path('/content/drive/MyDrive/claude/investment-agent')
    CACHE_DIR = Path('/content/tob_prediction')
else:
    PROJECT_ROOT = Path('C:/gdrive/claude/investment-agent')
    CACHE_DIR = Path('C:/tmp/tob_prediction')

CACHE_DIR.mkdir(parents=True, exist_ok=True)

# train_rf.py を import path に追加
sys.path.insert(0, str(PROJECT_ROOT / 'scripts' / 'tob_prediction'))

import numpy as np
import pandas as pd
import shap
import matplotlib
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from imblearn.under_sampling import RandomUnderSampler, TomekLinks
from imblearn.over_sampling import SMOTENC

# --- 日本語フォント ---
if RUNTIME == 'colab':
    import japanize_matplotlib
    matplotlib.rcParams['axes.unicode_minus'] = False
else:
    plt.rcParams["font.family"] = "MS Gothic"

# train_rf.py の関数・定数を再利用（CACHE_DIR を上書き）
import train_rf
train_rf.CACHE_DIR = CACHE_DIR
from train_rf import (
    build_feature_matrix,
    CONTINUOUS_FEATURES,
    BINARY_FEATURES,
    _cached, _bq_client,
    _load_labels, _load_financials, _load_shareholders,
    _load_price_features, _load_industries,
)

plt.rcParams["figure.figsize"] = (12, 8)
plt.rcParams["figure.dpi"] = 100
print(f"RUNTIME={RUNTIME}, CACHE_DIR={CACHE_DIR}")

## 1. データロード（キャッシュ再利用）

In [ ]:
# --- BQ client (runtime-dependent) ---
if RUNTIME == 'colab':
    auth.authenticate_user()
    from google.cloud import bigquery
    client = bigquery.Client(project='gmailpj-357912')
else:
    client = _bq_client()

# --- Load with cache (BQ fallback if cache miss) ---
labels = _cached("labels", _load_labels, client, refresh=False)
financials = _cached("financials", _load_financials, client, refresh=False)
shareholders = _cached("shareholders", _load_shareholders, client, refresh=False)
prices = _cached("prices", _load_price_features, client, refresh=False)
industries = _cached("industries", _load_industries, client, refresh=False)

print(f"Labels: {len(labels)}, Financials: {len(financials)}, Shareholders: {len(shareholders)}")
print(f"Prices: {len(prices)}, Industries: {len(industries)}")

features = build_feature_matrix(financials, shareholders, prices, industries, labels)
print(f"\nFeature matrix: {features.shape}")
print(f"Positives: {features['label'].sum():.0f} / {len(features)}")

## 2. モデル構築（2025年評価、訓練データ最大）

In [ ]:
EVAL_YEAR = 2025
TRAIN_WINDOW = 5

feature_cols = CONTINUOUS_FEATURES + BINARY_FEATURES + [
    c for c in features.columns if c.startswith("ind17_")
]
n_cont = len(CONTINUOUS_FEATURES)
cat_indices = list(range(n_cont, len(feature_cols)))

# 日本語の特徴量名マッピング
FEATURE_NAMES_JA = {
    "equity_ratio": "自己資本比率",
    "pbr": "PBR",
    "roe": "ROE(実績)",
    "payout_ratio": "配当性向",
    "ln_market_cap": "log(時価総額)",
    "cash_rich_ratio": "キャッシュリッチ比率",
    "forecast_div_yield": "予想配当利回り",
    "forecast_profit_growth": "予想利益成長率",
    "cfo_to_mcap": "CF/時価総額",
    "operating_margin": "営業利益率",
    "ret_60d": "60日リターン",
    "ret_240d": "240日リターン",
    "vol_240d": "240日ボラティリティ",
    "turnover_ratio": "出来高回転率",
    "top_shareholder_ratio": "筆頭株主比率",
    "individual_ratio": "個人持株比率",
    "foreign_ratio": "外国人持株比率",
    "financial_inst_ratio": "金融機関持株比率",
    "other_corp_ratio": "法人持株比率",
    "top10_concentration": "上位10株主集中度",
    "has_activist": "アクティビスト有無",
    "top_shareholder_is_public": "筆頭株主上場",
}
feature_names_display = [FEATURE_NAMES_JA.get(c, c) for c in feature_cols]

# Split
train_years = list(range(EVAL_YEAR - TRAIN_WINDOW, EVAL_YEAR))
df_train = features[features["year"].isin(train_years)].dropna(subset=feature_cols)
df_test = features[features["year"] == EVAL_YEAR].dropna(subset=feature_cols)

X_train = df_train[feature_cols].values.astype(np.float64)
y_train = df_train["label"].values.astype(int)
X_test = df_test[feature_cols].values.astype(np.float64)
y_test = df_test["label"].values.astype(int)

print(f"Train: {len(X_train)} (pos={y_train.sum()})")
print(f"Test:  {len(X_test)} (pos={y_test.sum()})")

# Standardize continuous features
scaler = StandardScaler()
X_train[:, :n_cont] = scaler.fit_transform(X_train[:, :n_cont])
X_test[:, :n_cont] = scaler.transform(X_test[:, :n_cont])

# Resample (6-step preprocessing)
rus = RandomUnderSampler(sampling_strategy=0.05, random_state=42)
X_r, y_r = rus.fit_resample(X_train, y_train)
tl = TomekLinks()
X_r, y_r = tl.fit_resample(X_r, y_r)
smote = SMOTENC(
    categorical_features=cat_indices,
    sampling_strategy=0.1,
    random_state=42,
    k_neighbors=min(5, int(y_r.sum()) - 1),
)
X_r, y_r = smote.fit_resample(X_r, y_r)
print(f"After resampling: {len(X_r)} (pos={y_r.sum()})")

# Train RF (2025 best params from walk-forward)
clf = RandomForestClassifier(
    n_estimators=300, max_depth=15, min_samples_leaf=14,
    max_features="sqrt", random_state=42, n_jobs=-1,
)
clf.fit(X_r, y_r)

from sklearn.metrics import roc_auc_score, average_precision_score
y_prob = clf.predict_proba(X_test)[:, 1]
print(f"\nROC-AUC: {roc_auc_score(y_test, y_prob):.4f}")
print(f"PR-AUC:  {average_precision_score(y_test, y_prob):.4f}")

## 3. SHAP値計算

In [ ]:
explainer = shap.TreeExplainer(clf)
shap_values = explainer.shap_values(X_test)

# RF の shap_values は [class0, class1] のリスト
# class1 (TOB=1) の SHAP 値を使う
if isinstance(shap_values, list):
    sv = shap_values[1]
else:
    sv = shap_values

print(f"SHAP values shape: {sv.shape}")
print(f"Test samples: {len(X_test)}, Features: {len(feature_cols)}")

## 4. SHAP Summary Plot（全特徴量の寄与度ビースウォーム）

In [ ]:
shap.summary_plot(sv, X_test, feature_names=feature_names_display, max_display=20, show=False)
plt.title("SHAP Summary — TOB予測 (2025年テスト)", fontsize=14)
plt.tight_layout()
plt.show()

## 5. SHAP Bar Plot（平均絶対SHAP値）

In [ ]:
shap.summary_plot(sv, X_test, feature_names=feature_names_display, plot_type="bar", max_display=20, show=False)
plt.title("特徴量重要度（平均|SHAP|）— TOB予測 (2025年)", fontsize=14)
plt.tight_layout()
plt.show()

## 6. Dependence Plots（論文の重要特徴量 Top4）

論文 SHAP 分析の重要変数: 筆頭株主比率 → 筆頭株主上場 → 個人持株比率 → log(時価総額)

In [ ]:
key_features = [
    ("top_shareholder_ratio", "筆頭株主比率"),
    ("top_shareholder_is_public", "筆頭株主上場"),
    ("individual_ratio", "個人持株比率"),
    ("ln_market_cap", "log(時価総額)"),
    ("other_corp_ratio", "法人持株比率"),
    ("has_activist", "アクティビスト有無"),
]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, (feat, ja_name) in zip(axes.flat, key_features):
    idx = feature_cols.index(feat)
    plt.sca(ax)
    shap.dependence_plot(
        idx, sv, X_test,
        feature_names=feature_names_display,
        show=False, ax=ax,
    )
    ax.set_title(f"{ja_name}", fontsize=12)

plt.suptitle("SHAP Dependence — 主要特徴量", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 7. HAS_ACTIVIST 固有寄与度分析

論文にない追加因子。アクティビスト保有がTOB予測にどう効くか。

In [ ]:
activist_idx = feature_cols.index("has_activist")
public_idx = feature_cols.index("top_shareholder_is_public")

# アクティビスト有無別の予測確率
mask_activist = X_test[:, activist_idx] == 1
mask_no_activist = X_test[:, activist_idx] == 0

print("=== HAS_ACTIVIST の効果 ===")
print(f"アクティビストあり: {mask_activist.sum()} 社, TOB実績 {y_test[mask_activist].sum():.0f} 件, "
      f"TOB率 {y_test[mask_activist].mean():.3%}")
print(f"アクティビストなし: {mask_no_activist.sum()} 社, TOB実績 {y_test[mask_no_activist].sum():.0f} 件, "
      f"TOB率 {y_test[mask_no_activist].mean():.3%}")
print(f"\n平均予測確率:")
print(f"  アクティビストあり: {y_prob[mask_activist].mean():.4f}")
print(f"  アクティビストなし: {y_prob[mask_no_activist].mean():.4f}")
print(f"\n平均SHAP値 (HAS_ACTIVIST):")
print(f"  アクティビストあり: {sv[mask_activist, activist_idx].mean():.6f}")
print(f"  アクティビストなし: {sv[mask_no_activist, activist_idx].mean():.6f}")
print(f"\n全特徴量中の重要度ランク: ", end="")
mean_abs_shap = np.abs(sv).mean(axis=0)
ranking = pd.Series(mean_abs_shap, index=feature_cols).sort_values(ascending=False)
rank = list(ranking.index).index("has_activist") + 1
print(f"{rank}位 / {len(feature_cols)}特徴量")

## 8. 論文比較テーブル（SHAP重要度ランキング）

In [ ]:
paper_ranking = {
    "top_shareholder_ratio": 1,
    "top_shareholder_is_public": 2,
    "individual_ratio": 3,
    "ln_market_cap": 4,
    "pbr": 5,
    "ret_240d": 6,
    "payout_ratio": 7,
}

comparison = []
for i, (feat, shap_val) in enumerate(ranking.items()):
    ja = FEATURE_NAMES_JA.get(feat, feat)
    paper_rank = paper_ranking.get(feat, "-")
    comparison.append({
        "実装ランク": i + 1,
        "特徴量": ja,
        "平均|SHAP|": f"{shap_val:.6f}",
        "論文ランク": paper_rank,
    })

comp_df = pd.DataFrame(comparison[:15])
print("=== SHAP重要度: 実装 vs 論文 ===")
print(comp_df.to_string(index=False))